# 🏋️ Apple Health Analyzer — Gradio Web UI

Run all cells top to bottom. The **last cell** will print a public URL — open it in any browser, upload your zip, and get your full dashboard instantly.



In [1]:
!pip install -q gradio plotly scikit-learn scipy statsmodels

In [2]:
import zipfile, warnings, json, io, os, traceback
import xml.etree.ElementTree as ET
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import gradio as gr

warnings.filterwarnings('ignore')
print('All imports OK.')

All imports OK.


In [3]:
TARGET = {
    'HKQuantityTypeIdentifierHeartRateVariabilitySDNN': 'hrv',
    'HKQuantityTypeIdentifierRestingHeartRate':          'rhr',
    'HKQuantityTypeIdentifierHeartRate':                 'hr',
    'HKQuantityTypeIdentifierActiveEnergyBurned':        'active_cal',
    'HKQuantityTypeIdentifierBasalEnergyBurned':         'basal_cal',
    'HKQuantityTypeIdentifierStepCount':                 'steps',
    'HKQuantityTypeIdentifierAppleExerciseTime':         'exercise_min',
    'HKQuantityTypeIdentifierVO2Max':                    'vo2max',
    'HKQuantityTypeIdentifierOxygenSaturation':          'spo2',
    'HKQuantityTypeIdentifierRespiratoryRate':            'resp_rate',
    'HKQuantityTypeIdentifierWalkingSpeed':               'walk_speed',
    'HKQuantityTypeIdentifierBodyMass':                  'weight',
    'HKQuantityTypeIdentifierAppleSleepingWristTemperature': 'wrist_temp',
    'HKQuantityTypeIdentifierDistanceWalkingRunning':    'distance',
    'HKQuantityTypeIdentifierFlightsClimbed':            'flights',
}
SLEEP_MAP = {
    'HKCategoryValueSleepAnalysisAsleepDeep': 'deep',
    'HKCategoryValueSleepAnalysisAsleepCore': 'core',
    'HKCategoryValueSleepAnalysisAsleepREM':  'rem',
    'HKCategoryValueSleepAnalysisAsleep':     'asleep',
    'HKCategoryValueSleepAnalysisInBed':      'inbed',
    'HKCategoryValueSleepAnalysisAwake':      'awake',
}

def to_date(s):
    try: return datetime.strptime(s[:10], '%Y-%m-%d').date()
    except: return None

def to_dt(s):
    try: return datetime.strptime(s[:19], '%Y-%m-%d %H:%M:%S')
    except: return None

def parse_zip(zip_path):
    print('Parsing XML...')
    with open(zip_path, 'rb') as f:
        zip_bytes = f.read()
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        with z.open('apple_health_export/export.xml') as f:
            tree = ET.parse(f)
    root = tree.getroot()
    data = defaultdict(lambda: defaultdict(list))
    sleep_records, workout_records = [], []
    for rec in root.iter('Record'):
        rtype = rec.get('type','')
        val   = rec.get('value','')
        start = rec.get('startDate','')
        end   = rec.get('endDate','')
        d = to_date(start)
        if d is None: continue
        if rtype in TARGET:
            try: data[d][TARGET[rtype]].append(float(val))
            except: pass
        elif rtype == 'HKCategoryTypeIdentifierSleepAnalysis':
            stage = SLEEP_MAP.get(val)
            if stage:
                s_dt, e_dt = to_dt(start), to_dt(end)
                if s_dt and e_dt:
                    sleep_records.append({'date': d, 'stage': stage,
                                          'hours': (e_dt-s_dt).total_seconds()/3600})
    for w in root.iter('Workout'):
        d = to_date(w.get('startDate',''))
        if not d: continue
        dur   = float(w.get('duration', 0) or 0)
        wtype = w.get('workoutActivityType','').replace('HKWorkoutActivityType','')
        avg_hr = None
        for stat in w.iter('WorkoutStatistics'):
            if 'HeartRate' in stat.get('type',''):
                try: avg_hr = float(stat.get('average') or 0)
                except: pass
        workout_records.append({'date': d, 'type': wtype, 'duration_min': dur, 'avg_hr': avg_hr})
    print(f'Parsed: {len(data)} days, {len(sleep_records)} sleep, {len(workout_records)} workouts')
    return data, sleep_records, workout_records

print('Parser ready.')

Parser ready.


In [4]:
def build_dataframe(data, sleep_records, workout_records):
    rows = []
    for d, vals in sorted(data.items()):
        row = {'date': d}
        for k in TARGET.values():
            if k in vals:
                v = vals[k]
                if k in ['rhr','vo2max','wrist_temp']: row[k] = v[-1]
                elif k in ['active_cal','basal_cal','steps','exercise_min','distance','flights']: row[k] = sum(v)
                else: row[k] = float(np.mean(v))
            else: row[k] = np.nan
        rows.append(row)
    df = pd.DataFrame(rows)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    if sleep_records:
        sdf = pd.DataFrame(sleep_records)
        sdf['date'] = pd.to_datetime(sdf['date'])
        spiv = sdf.groupby(['date','stage'])['hours'].sum().unstack(fill_value=0)
        for c in ['deep','core','rem','awake','inbed']:
            if c not in spiv: spiv[c] = 0
        spiv['total_sleep'] = spiv[['deep','core','rem']].sum(axis=1)
        if 'asleep' in spiv.columns:
            spiv['total_sleep'] = spiv['total_sleep'].where(spiv['total_sleep']>0, spiv['asleep'])
        df = df.merge(spiv[['deep','core','rem','awake','total_sleep']].reset_index(), on='date', how='left')
    if workout_records:
        wdf = pd.DataFrame(workout_records)
        wdf['date'] = pd.to_datetime(wdf['date'])
        wg = wdf.groupby('date').apply(lambda x: pd.Series({
            'workout_count':  len(x),
            'total_duration': x['duration_min'].sum(),
            'workout_load':   (x['duration_min'] * (x['avg_hr'].fillna(130)/170)).sum(),
            'workout_types':  ', '.join(x['type'].unique())
        })).reset_index()
        df = df.merge(wg, on='date', how='left')
        df['workout_load']   = df['workout_load'].fillna(0)
        df['total_duration'] = df['total_duration'].fillna(0)
        df['workout_count']  = df['workout_count'].fillna(0)
    dfw = df[df['date'] >= '2025-01-01'].copy().reset_index(drop=True)
    if len(dfw) < 20:
        dfw = df.copy().reset_index(drop=True)
    return dfw

print('DataFrame builder ready.')

DataFrame builder ready.


In [5]:
def run_analysis(dfw):
    dfw['hrv_roll_mean'] = dfw['hrv'].rolling(28, min_periods=5).mean()
    dfw['hrv_roll_std']  = dfw['hrv'].rolling(28, min_periods=5).std().clip(lower=1)
    dfw['rhr_roll_mean'] = dfw['rhr'].rolling(28, min_periods=5).mean()
    dfw['rhr_roll_std']  = dfw['rhr'].rolling(28, min_periods=5).std().clip(lower=1)
    dfw['hrv_z'] = (dfw['hrv'] - dfw['hrv_roll_mean']) / dfw['hrv_roll_std']
    dfw['rhr_z'] = (dfw['rhr_roll_mean'] - dfw['rhr']) / dfw['rhr_roll_std']
    if 'total_sleep' in dfw.columns:
        deep_c = dfw['deep'].fillna(0) if 'deep' in dfw.columns else pd.Series(0, index=dfw.index)
        rem_c  = dfw['rem'].fillna(0)  if 'rem'  in dfw.columns else pd.Series(0, index=dfw.index)
        dfw['sleep_q'] = (
            (dfw['total_sleep'].fillna(0)/8).clip(0,1)*0.5 +
            (deep_c/1.5).clip(0,1)*0.25 + (rem_c/2.0).clip(0,1)*0.25
        )
    else:
        dfw['sleep_q'] = 0.5
    dfw['recovery'] = (
        50 + dfw['hrv_z'].fillna(0)*0.50*20
           + dfw['rhr_z'].fillna(0)*0.30*20
           + (dfw['sleep_q'].fillna(0.5)-0.5)*0.20*20
    ).clip(0,100).round(1)
    label_fn = lambda v: 'Peak' if v>=70 else 'Good' if v>=55 else 'Moderate' if v>=40 else 'Low'
    dfw['rec_label'] = dfw['recovery'].apply(label_fn)
    for f in ['active_cal','exercise_min','steps','workout_load']:
        if f not in dfw.columns: dfw[f] = 0
    dfw_ef = dfw[['active_cal','exercise_min','steps','workout_load']].fillna(0).copy()
    for f in dfw_ef.columns:
        cap = dfw_ef[f].quantile(0.99)
        if cap > 0: dfw_ef[f] = dfw_ef[f].clip(upper=cap)
    sc = MinMaxScaler()
    dfw[['cal_n','ex_n','step_n','load_n']] = sc.fit_transform(dfw_ef)
    dfw['energy'] = (dfw['cal_n']*0.35+dfw['ex_n']*0.35+dfw['step_n']*0.15+dfw['load_n']*0.15).clip(0,1)*100
    dfw['acute']   = dfw['workout_load'].rolling(7,  min_periods=1).mean()
    dfw['chronic'] = dfw['workout_load'].rolling(28, min_periods=7).mean()
    dfw['acwr']    = (dfw['acute'] / dfw['chronic'].clip(lower=1)).round(3)
    dfw['acwr_zone'] = dfw['acwr'].apply(
        lambda v: 'Undertraining' if v<0.8 else 'Sweet Spot' if v<=1.3 else 'Caution' if v<=1.5 else 'High Risk'
    )
    def hrv_slope(s):
        s = s.dropna()
        if len(s)<3: return 0
        return float(np.polyfit(np.arange(len(s)), s.values, 1)[0])
    dfw['hrv_7d_slope'] = dfw['hrv'].rolling(7, min_periods=3).apply(hrv_slope)
    dfw['rec_7d_avg']   = dfw['recovery'].rolling(7, min_periods=3).mean()
    dfw['rec_trend']    = dfw['recovery'].rolling(3, min_periods=2).mean() - dfw['rec_7d_avg']
    def days_since_heavy(idx):
        for i in range(idx, -1, -1):
            if dfw['workout_load'].iloc[i] >= 60: return idx-i
        return 99
    dfw['days_since_heavy'] = [days_since_heavy(i) for i in range(len(dfw))]
    dfw['peak_score'] = (
        dfw['acwr'].apply(lambda v: 30*max(0,1-abs(v-1.05)/0.5)) +
        (dfw['recovery']/100*25) +
        dfw['rec_trend'].fillna(0).apply(lambda v: max(0,min(20,v*2))) +
        dfw['hrv_7d_slope'].fillna(0).apply(lambda v: max(0,min(15,v*3))) +
        dfw['days_since_heavy'].apply(lambda v: 10 if 1<=v<=3 else 0)
    ).clip(0,100).round(1)
    z_df = pd.DataFrame(index=dfw.index)
    for feat in ['hrv','rhr','active_cal','steps']:
        col = dfw[feat].fillna(dfw[feat].median()) if feat in dfw.columns else pd.Series(0,index=dfw.index)
        rm  = col.rolling(28, min_periods=7).mean()
        rs  = col.rolling(28, min_periods=7).std().clip(lower=0.5)
        z   = (col - rm) / rs
        z_df[feat] = z if feat=='rhr' else -z
    dfw['anomaly_score'] = z_df.clip(lower=0).mean(axis=1).round(3)
    dfw['anomaly'] = dfw['anomaly_score'] > 1.2
    cf = [c for c in ['hrv','rhr','active_cal','steps','exercise_min','recovery'] if c in dfw.columns]
    X  = dfw[cf].fillna(dfw[cf].median())
    sc2 = MinMaxScaler()
    Xs  = sc2.fit_transform(X)
    km  = KMeans(n_clusters=4, random_state=42, n_init=10)
    dfw['cluster'] = km.fit_predict(Xs)
    cm_means = dfw.groupby('cluster')[cf + ['workout_load']].mean()
    clabels = {}
    for cid, row in cm_means.iterrows():
        rec = row.get('recovery', 50); cal = row.get('active_cal', 0)
        if rec>=60 and cal>500:   clabels[cid]='High Performance'
        elif rec>=55 and cal<=500: clabels[cid]='Active Recovery'
        elif rec<45:               clabels[cid]='Fatigued'
        else:                      clabels[cid]='Moderate Training'
    dfw['day_type'] = dfw['cluster'].map(clabels)
    pca = PCA(n_components=2)
    Xp  = pca.fit_transform(Xs)
    dfw['pc1'] = Xp[:,0]
    dfw['pc2'] = Xp[:,1]
    pca_var = pca.explained_variance_ratio_.tolist()
    corr_cols = ['hrv','rhr','active_cal','steps','exercise_min','recovery','energy','workout_load','acwr','peak_score']
    valid_cols = [c for c in corr_cols if c in dfw.columns and dfw[c].notna().sum()>=10]
    corr_m = dfw[valid_cols].corr(method='spearman').round(3)
    return dfw, corr_m, pca_var

print('Analysis pipeline ready.')

Analysis pipeline ready.


In [6]:
def build_charts(dfw, corr_m, pca_var):
    charts = {}
    d90 = dfw.tail(90)
    fig1 = make_subplots(rows=3, cols=1, shared_xaxes=True,
        subplot_titles=['Recovery Score','Energy Score','Peak Performance Score'],
        vertical_spacing=0.08)
    spec = [('recovery','#00c896','rgba(0,200,150,0.1)'),
            ('energy',  '#f5c518','rgba(245,197,24,0.1)'),
            ('peak_score','#ff6b6b','rgba(255,107,107,0.1)')]
    for row,(col,clr,fill) in enumerate(spec,1):
        fig1.add_trace(go.Scatter(x=d90['date'],y=d90[col],mode='lines',
            line=dict(color=clr,width=2),fill='tozeroy',fillcolor=fill,
            name=col,hovertemplate='%{x|%b %d}: <b>%{y:.1f}</b><extra></extra>'),row=row,col=1)
        wd = d90[d90.get('workout_load',pd.Series(0,index=d90.index))>10]
        fig1.add_trace(go.Scatter(x=wd['date'],y=wd[col],mode='markers',
            marker=dict(color=clr,size=5,symbol='circle-open'),
            showlegend=(row==1),name='Workout Day'),row=row,col=1)
    fig1.update_layout(title='Recovery / Energy / Peak — 90-Day Timeline',
        height=600,template='plotly_dark',showlegend=False,
        paper_bgcolor='#0d1117',plot_bgcolor='#0d1117')
    fig1.update_yaxes(range=[0,105],gridcolor='#1c2128')
    fig1.update_xaxes(gridcolor='#1c2128')
    charts['timeline'] = fig1

    hrv_mean = float(dfw['hrv'].mean())
    hrv_std  = float(dfw['hrv'].std())
    cmap = {'Peak':'#00c896','Good':'#64c896','Moderate':'#f5c518','Low':'#ff4444'}
    fig2 = go.Figure()
    fig2.add_hrect(y0=hrv_mean-hrv_std,y1=hrv_mean+hrv_std,
        fillcolor='rgba(0,200,150,0.08)',line_width=0,
        annotation_text='Normal Range',annotation_position='top right')
    fig2.add_hline(y=hrv_mean,line_dash='dash',line_color='rgba(0,200,150,0.4)')
    for label,color in cmap.items():
        sub = dfw[dfw['rec_label']==label]
        if len(sub)==0: continue
        fig2.add_trace(go.Scatter(x=sub['date'],y=sub['hrv'],mode='markers',name=label,
            marker=dict(color=color,size=7,opacity=0.8),
            hovertemplate='%{x|%b %d} HRV: %{y:.1f} ms<extra></extra>'))
    fig2.add_trace(go.Scatter(x=dfw['date'],y=dfw['hrv'].rolling(7).mean(),
        mode='lines',name='7-Day Avg',line=dict(color='white',width=2)))
    fig2.update_layout(title='HRV Daily Values — Colored by Recovery Level',
        height=420,template='plotly_dark',
        paper_bgcolor='#0d1117',plot_bgcolor='#0d1117')
    charts['hrv'] = fig2

    d60 = dfw.tail(60)
    fig3 = make_subplots(rows=2,cols=1,shared_xaxes=True,
        subplot_titles=['Training Load','ACWR — Injury Risk'],vertical_spacing=0.12)
    fig3.add_trace(go.Bar(x=d60['date'],y=d60.get('workout_load',pd.Series(0,index=d60.index)),
        name='Daily Load',marker_color='rgba(100,160,255,0.4)'),row=1,col=1)
    if 'acute' in d60.columns:
        fig3.add_trace(go.Scatter(x=d60['date'],y=d60['acute'],name='Acute 7d',
            line=dict(color='#00c896',width=2)),row=1,col=1)
    if 'chronic' in d60.columns:
        fig3.add_trace(go.Scatter(x=d60['date'],y=d60['chronic'],name='Chronic 28d',
            line=dict(color='#f5c518',width=2)),row=1,col=1)
    fig3.add_hrect(y0=0.8,y1=1.3,row=2,col=1,fillcolor='rgba(0,200,150,0.12)',line_width=0)
    fig3.add_hrect(y0=1.3,y1=1.5,row=2,col=1,fillcolor='rgba(245,197,24,0.1)',line_width=0)
    fig3.add_hrect(y0=1.5,y1=3.0,row=2,col=1,fillcolor='rgba(255,107,107,0.08)',line_width=0)
    fig3.add_trace(go.Scatter(x=d60['date'],y=d60['acwr'],name='ACWR',
        line=dict(color='white',width=2)),row=2,col=1)
    fig3.update_layout(title='Training Load & ACWR — Injury Risk',
        height=500,template='plotly_dark',
        paper_bgcolor='#0d1117',plot_bgcolor='#0d1117')
    charts['acwr'] = fig3

    labels_clean = [c.replace('_',' ').title() for c in corr_m.columns]
    fig4 = go.Figure(go.Heatmap(
        z=corr_m.values,x=labels_clean,y=labels_clean,
        colorscale=[[0,'#ff4444'],[0.5,'#1c2128'],[1,'#00c896']],
        zmid=0,zmin=-1,zmax=1,text=corr_m.values.round(2),
        texttemplate='%{text}',textfont={'size':9},
        hovertemplate='%{y} x %{x}: <b>%{z:.2f}</b><extra></extra>',
        colorbar=dict(title='Spearman r')
    ))
    fig4.update_layout(title='Spearman Correlation Matrix',
        height=500,template='plotly_dark',
        paper_bgcolor='#0d1117',plot_bgcolor='#0d1117')
    charts['correlation'] = fig4

    cmap2 = {'High Performance':'#00c896','Active Recovery':'#64a0ff',
             'Moderate Training':'#f5c518','Fatigued':'#ff4444'}
    fig5 = go.Figure()
    for dtype,color in cmap2.items():
        sub = dfw[dfw['day_type']==dtype] if 'day_type' in dfw.columns else pd.DataFrame()
        if len(sub)==0: continue
        fig5.add_trace(go.Scatter(
            x=sub['pc1'],y=sub['pc2'],mode='markers',name=dtype,
            marker=dict(color=color,size=8,opacity=0.8),
            customdata=sub[['date','recovery','hrv']].values,
            hovertemplate='<b>'+dtype+'</b><br>%{customdata[0]}<br>Recovery: %{customdata[1]:.0f}<extra></extra>'
        ))
    pv = pca_var
    fig5.update_layout(
        title=f'Day-Type Clustering (PCA) — PC1+PC2 = {(pv[0]+pv[1])*100:.0f}% variance',
        xaxis_title=f'PC1 ({pv[0]*100:.0f}%)',yaxis_title=f'PC2 ({pv[1]*100:.0f}%)',
        height=450,template='plotly_dark',
        paper_bgcolor='#0d1117',plot_bgcolor='#0d1117')
    charts['clustering'] = fig5

    fig6 = go.Figure()
    fig6.add_trace(go.Scatter(x=dfw['date'],y=dfw['recovery'],mode='lines',
        fill='tozeroy',line=dict(color='rgba(100,160,255,0.5)',width=1),
        fillcolor='rgba(100,160,255,0.05)',name='Recovery'))
    anom = dfw[dfw['anomaly']] if 'anomaly' in dfw.columns else pd.DataFrame()
    if len(anom):
        fig6.add_trace(go.Scatter(x=anom['date'],y=anom['recovery'],mode='markers',
            name='Anomaly',marker=dict(color='#ff4444',size=12,symbol='x',
                                       line=dict(width=2,color='#ff4444')),
            hovertemplate='ANOMALY %{x|%b %d}<br>Recovery: %{y:.0f}<extra></extra>'))
    fig6.update_layout(title='Anomaly Detection — Illness & Overtraining Signals',
        height=400,template='plotly_dark',
        paper_bgcolor='#0d1117',plot_bgcolor='#0d1117')
    charts['anomaly'] = fig6

    dfw_c = dfw.copy()
    dfw_c['week'] = dfw_c['date'].dt.isocalendar().week.astype(int)
    dfw_c['dow']  = dfw_c['date'].dt.dayofweek
    cal_p = dfw_c.pivot_table(index='dow',columns='week',values='recovery',aggfunc='mean')
    dow_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    fig7 = go.Figure(go.Heatmap(
        z=cal_p.values,x=cal_p.columns.astype(str),
        y=[dow_labels[i] for i in cal_p.index],
        colorscale=[[0,'#ff4444'],[0.4,'#f5c518'],[0.7,'#00c896'],[1,'#00ffc8']],
        zmid=50,zmin=0,zmax=100,
        hovertemplate='Week %{x}, %{y}: <b>%{z:.0f}</b><extra></extra>',
        colorbar=dict(title='Recovery')))
    fig7.update_layout(title='Recovery Calendar Heatmap',
        height=350,template='plotly_dark',
        paper_bgcolor='#0d1117',plot_bgcolor='#0d1117')
    charts['calendar'] = fig7

    if 'vo2max' in dfw.columns:
        vo2_data = dfw[dfw['vo2max'].notna()][['date','vo2max']]
        if len(vo2_data) > 0:
            fig8 = go.Figure()
            fig8.add_hrect(y0=37,y1=42,fillcolor='rgba(100,160,255,0.08)',line_width=0)
            fig8.add_hrect(y0=42,y1=60,fillcolor='rgba(0,200,150,0.06)',line_width=0)
            fig8.add_trace(go.Scatter(x=vo2_data['date'],y=vo2_data['vo2max'],
                mode='lines+markers+text',line=dict(color='#64a0ff',width=2),
                marker=dict(color='#64a0ff',size=12),
                text=vo2_data['vo2max'].round(1).astype(str),textposition='top center',
                hovertemplate='%{x|%b %d}: VO2 Max <b>%{y:.2f}</b><extra></extra>'))
            fig8.update_layout(title='VO2 Max Trend — Aerobic Fitness',
                height=350,template='plotly_dark',
                paper_bgcolor='#0d1117',plot_bgcolor='#0d1117')
            charts['vo2max'] = fig8
    return charts

print('Chart builder ready.')

Chart builder ready.


In [7]:
def build_summary(dfw):
    t    = dfw.iloc[-1]
    l7   = dfw.tail(7)
    l30  = dfw.tail(30)
    consec = 0
    for i in range(len(dfw)-1,-1,-1):
        if dfw.get('workout_load', pd.Series(0,index=dfw.index)).iloc[i] > 10: consec += 1
        else: break
    acwr_now = float(t.get('acwr', 1.0))
    rec_now  = float(t.get('recovery', 50))
    hrv_slope_now = float(t.get('hrv_7d_slope', 0)) if not pd.isna(t.get('hrv_7d_slope', 0)) else 0
    if rec_now>=70 and acwr_now<=1.3 and hrv_slope_now>=0:
        window = 'TODAY — Peak condition'
        action = 'GO HARD: Max intensity session, push for PRs'
        window_color = '#00c896'
    elif rec_now>=55 and acwr_now<=1.4:
        window = '1-2 DAYS - Building phase'
        action = 'TRAIN HARD: High volume at 80-90% effort'
        window_color = '#f5c518'
    elif rec_now<40 or consec>=4:
        window = '2-4 DAYS - After recovery'
        action = 'REST: Light walk or yoga only. Sleep is your workout.'
        window_color = '#ff4444'
    else:
        window = '2-3 DAYS - After moderate build'
        action = 'MODERATE: Zone 2 cardio, build the base'
        window_color = '#ff8c00'
    def safe(v):
        if v is None: return None
        if isinstance(v, float) and (v != v): return None
        try: return float(v)
        except: return str(v)
    return {
        'today': {
            'date':        str(t['date'].date()),
            'recovery':    safe(t.get('recovery')),
            'energy':      safe(t.get('energy')),
            'peak_score':  safe(t.get('peak_score')),
            'hrv':         safe(t.get('hrv')),
            'rhr':         safe(t.get('rhr')),
            'steps':       safe(t.get('steps')),
            'active_cal':  safe(t.get('active_cal')),
            'exercise_min':safe(t.get('exercise_min')),
            'acwr':        safe(acwr_now),
            'acwr_zone':   str(t.get('acwr_zone','--')),
        },
        'trends': {
            'avg_recovery_7d':  safe(l7['recovery'].mean()),
            'avg_hrv_7d':       safe(l7['hrv'].mean()),
            'workout_days_30d': int((l30.get('workout_load', pd.Series(0,index=l30.index))>10).sum()),
            'peak_days_total':  int((dfw['recovery']>=70).sum()),
            'low_days_total':   int((dfw['recovery']<40).sum()),
            'anomaly_days':     int(dfw.get('anomaly', pd.Series(False,index=dfw.index)).sum()),
        },
        'recommendation': {
            'window': window,
            'action': action,
            'color':  window_color,
        },
        'total_days': len(dfw),
    }

print('Summary builder ready.')

Summary builder ready.


In [8]:
def analyze_health(zip_file):
    if zip_file is None:
        return ('<p style="color:#ff4444">Please upload a zip file.</p>',) + (None,)*8
    try:
        data, sleep_records, workout_records = parse_zip(zip_file.name)
        dfw = build_dataframe(data, sleep_records, workout_records)
        dfw, corr_m, pca_var = run_analysis(dfw)
        charts  = build_charts(dfw, corr_m, pca_var)
        summary = build_summary(dfw)
        t = summary['today']
        tr = summary['trends']
        rec = summary['recommendation']
        cards = ''.join([
            f'<div style="background:#161b22;padding:12px;border-radius:8px;text-align:center">'
            f'<div style="font-size:22px;font-weight:bold;color:{color}">{val:.0f}</div>'
            f'<div style="font-size:11px;color:#8b949e">{label}</div></div>'
            for label,val,color in [
                ('Recovery',   t['recovery']   or 0, '#00c896'),
                ('Energy',     t['energy']     or 0, '#f5c518'),
                ('Peak Score', t['peak_score'] or 0, '#ff6b6b'),
                ('HRV (ms)',   t['hrv']        or 0, '#64a0ff'),
            ] if val is not None
        ])
        stats = ''.join([
            f'<div style="background:#161b22;padding:10px;border-radius:8px;text-align:center">'
            f'<div style="font-size:18px;font-weight:bold;color:#c9d1d9">{val:.0f}</div>'
            f'<div style="font-size:11px;color:#8b949e">{label}</div></div>'
            for label,val in [
                ('RHR (bpm)',    t['rhr']         or 0),
                ('Steps',       t['steps']       or 0),
                ('Active Cal',  t['active_cal']  or 0),
                ('Exercise min',t['exercise_min'] or 0),
            ] if val is not None
        ])
        summary_html = f"""
        <div style='background:#0d1117;color:#e6edf3;font-family:sans-serif;padding:20px;border-radius:12px;border:1px solid #30363d'>
          <h2 style='color:#00c896;margin:0 0 4px 0'>Today: {t['date']}</h2>
          <p style='color:#8b949e;margin:0 0 16px 0'>{summary['total_days']} days of data analyzed</p>
          <div style='display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-bottom:16px'>{cards}</div>
          <div style='display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-bottom:16px'>{stats}</div>
          <div style='background:{rec["color"]}22;border:1px solid {rec["color"]}55;border-radius:8px;padding:14px'>
            <div style='color:{rec["color"]};font-weight:bold;font-size:14px'>Target: {rec["window"]}</div>
            <div style='color:#c9d1d9;margin-top:4px'>{rec["action"]}</div>
          </div>
          <div style='margin-top:14px;display:grid;grid-template-columns:repeat(3,1fr);gap:10px'>
            <div style='background:#161b22;padding:10px;border-radius:8px;text-align:center'>
              <div style='font-size:18px;font-weight:bold;color:#c9d1d9'>{tr['avg_recovery_7d']:.0f}</div>
              <div style='font-size:11px;color:#8b949e'>Avg Recovery 7d</div>
            </div>
            <div style='background:#161b22;padding:10px;border-radius:8px;text-align:center'>
              <div style='font-size:18px;font-weight:bold;color:#c9d1d9'>{tr['workout_days_30d']}</div>
              <div style='font-size:11px;color:#8b949e'>Workouts last 30d</div>
            </div>
            <div style='background:#161b22;padding:10px;border-radius:8px;text-align:center'>
              <div style='font-size:18px;font-weight:bold;color:#ff4444'>{tr['anomaly_days']}</div>
              <div style='font-size:11px;color:#8b949e'>Anomaly Days</div>
            </div>
          </div>
        </div>
        """
        return (
            summary_html,
            charts.get('timeline'),
            charts.get('hrv'),
            charts.get('acwr'),
            charts.get('correlation'),
            charts.get('clustering'),
            charts.get('anomaly'),
            charts.get('calendar'),
            charts.get('vo2max'),
        )
    except Exception as e:
        import traceback
        err = traceback.format_exc()
        print(err)
        error_html = f'<div style="color:#ff4444;padding:20px;background:#0d1117;border-radius:8px"><b>Error:</b><pre style="white-space:pre-wrap">{err}</pre></div>'
        return (error_html,) + (None,)*8

print('Gradio handler ready.')

Gradio handler ready.


In [9]:
with gr.Blocks(
    theme=gr.themes.Base(primary_hue='emerald', neutral_hue='slate'),
    css='body, .gradio-container { background: #0d1117 !important; }',
    title='Apple Health Analyzer'
) as demo:

    gr.Markdown("""
    # Apple Health Analyzer
    Export from iPhone Health app: **Share** > **Export All Health Data**.
    Upload the zip below to get your full dashboard.
    """)

    with gr.Row():
        file_input = gr.File(label='Upload apple_health_export.zip', file_types=['.zip'])
        analyze_btn = gr.Button('Analyze', variant='primary', scale=0)

    summary_out = gr.HTML(label='Summary')

    with gr.Tabs():
        with gr.Tab('Timeline'):       chart1 = gr.Plot()
        with gr.Tab('HRV'):            chart2 = gr.Plot()
        with gr.Tab('Training Load'):  chart3 = gr.Plot()
        with gr.Tab('Correlations'):   chart4 = gr.Plot()
        with gr.Tab('Clustering'):     chart5 = gr.Plot()
        with gr.Tab('Anomalies'):      chart6 = gr.Plot()
        with gr.Tab('Calendar'):       chart7 = gr.Plot()
        with gr.Tab('VO2 Max'):        chart8 = gr.Plot()

    outputs = [summary_out, chart1, chart2, chart3, chart4, chart5, chart6, chart7, chart8]

    analyze_btn.click(fn=analyze_health, inputs=[file_input], outputs=outputs)
    file_input.change(fn=analyze_health, inputs=[file_input], outputs=outputs)

# share=True gives a public URL valid for 72 hours
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2ea779131728569c3b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
